In [ ]:
!pip install torchmetrics
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader
from torchmetrics.classification import BinaryAUROC
from torch.optim import Adam
import tqdm
import os
import time
from datetime import datetime, timedelta
from google.colab import drive
drive.mount('/content/drive/')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 62.8 MB/s eta 0:00:00
Mounted at /content/drive/


In [ ]:
# Constants
BATCH_SIZE = 32
VOCAB_SIZE = 49152
EMBEDDING_SIZE = 4096
LSTM_NODES = 256
OUTPUT_DIM = 1
LEARNING_RATE = 0.001
EPOCHS = 20

# Check if CUDA is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Load tokenizer and extract embeddings from HuggingFace model
print("Loading pretrained embeddings from aiXcoder...")
tokenizer = AutoTokenizer.from_pretrained("aiXcoder/aixcoder-7b-base")
hf_model = AutoModelForCausalLM.from_pretrained("aiXcoder/aixcoder-7b-base")

# Extract token embeddings
word_vectors = hf_model.model.embed_tokens.weight.data.clone()
print(f"Loaded embeddings shape: {word_vectors.shape}")

# Clean up HuggingFace model to save memory
del hf_model
torch.cuda.empty_cache() if torch.cuda.is_available() else None

# Paths to pretraining data (adjust these for your Colab setup)
c_pretraining_path = '/content/drive/MyDrive/romeo/pretraining/c'
java_pretraining_path = '/content/drive/MyDrive/romeo/pretraining/java'
csharp_pretraining_path = '/content/drive/MyDrive/romeo/pretraining/csharp'
combined_path = '/content/drive/MyDrive/romeo/pretraining/combined'
output_path = '/content/drive/MyDrive/romeo/models'

# Create output directory
os.makedirs(output_path, exist_ok=True)

Using device: cuda
Loading pretrained embeddings from aiXcoder...
Loaded embeddings shape: torch.Size([49152, 4096])


Running mini-batch on LSTM

In [ ]:
!python /content/drive/MyDrive/romeo/load_and_predict.py



Using device: cuda
Loading pretrained embeddings from aiXcoder...
2025-12-05 20:47:51.368092: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764967671.387048   17944 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764967671.393631   17944 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1764967671.410261   17944 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1764967671.410294   17944 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:176496767

In [ ]:
!python /content/drive/MyDrive/romeo/inference_example.py

Using device: cuda

EVALUATION MATRIX: Models vs Test Sets

Model: C
Loaded model: c
Best Val Loss: 0.0000
Final Val AUROC: 1.0000

  Testing on C test set...
    Samples: 361, Positive: 361
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
    Accuracy:  0.4266
    Precision: 1.0000
    Recall:    0.4266
    F1:        0.5981
    AUROC:     nan

  Testing on Python test set...
    Samples: 407, Positive: 407
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
    Accuracy:  0.5012
    Precision: 1.0000
    Recall:    0.5012
    F1:        0.6678
    AUROC:     nan

Model: COMBINED
Loaded model: combined
Best Val Loss: 0.0000
Final Val AUROC: 1.0000

  Testing on C test set...
    Samples: 361, Positive: 

Starting larger test with 10k dataset size.

In [ ]:
!python /content/drive/MyDrive/romeo/load_and_predict.py



Using device: cuda
Loading pretrained embeddings from aiXcoder...
tokenizer_config.json: 100% 925/925 [00:00<00:00, 6.92MB/s]
tokenizer.json: 3.06MB [00:00, 84.5MB/s]
tokenizer.model: 100% 871k/871k [00:01<00:00, 641kB/s]
special_tokens_map.json: 100% 3.00/3.00 [00:00<00:00, 20.9kB/s]
config.json: 100% 608/608 [00:00<00:00, 4.90MB/s]
2025-12-08 03:23:03.179465: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-08 03:23:03.197883: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1765164183.217303    2104 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already b

Running inference from the 10k models on the 10k evaluation set.

In [ ]:
!python /content/drive/MyDrive/romeo/inference_example.py

Using device: cuda

EVALUATION MATRIX: Models vs Test Sets

Model: C
Loaded model: c
Best Val Loss: 0.0000
Final Val AUROC: 1.0000

  Testing on C test set...
    Samples: 3712, Positive: 3712
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
    Accuracy:  0.6404
    Precision: 1.0000
    Recall:    0.6404
    F1:        0.7808
    AUROC:     nan

  Testing on Python test set...
    Samples: 577, Positive: 577
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
    Accuracy:  0.4939
    Precision: 1.0000
    Recall:    0.4939
    F1:        0.6613
    AUROC:     nan

Model: COMBINED
Loaded model: combined
Best Val Loss: 0.0000
Final Val AUROC: 1.0000

  Testing on C test set...
    Samples: 3712, Positiv

Making the following changes before the final inference run - Increased evaluation dataset size, removed AUROC, adding mulitiple seed inference trials to compute average and range of accuracy + other metrics.

Saved c full dataset: 18673 samples
Saved python full dataset: 2762 samples

In [ ]:
!python /content/drive/MyDrive/romeo/inference_example.py

Using device: cuda

EVALUATION MATRIX: Models vs Test Sets
Running 5 evaluations per model-dataset pair

Model: C
Loaded model: c
Best Val Loss: 0.0000
Final Val AUROC: 1.0000

  Testing on C test set...
    Samples: 18673, Positive: 18673
    Accuracy:  0.6377 ± 0.0000 (range: 0.0000)
    Precision: 1.0000 ± 0.0000 (range: 0.0000)
    Recall:    0.6377 ± 0.0000 (range: 0.0000)
    F1:        0.7788 ± 0.0000 (range: 0.0000)

  Testing on PYTHON test set...
    Samples: 2762, Positive: 2762
    Accuracy:  0.4899 ± 0.0000 (range: 0.0000)
    Precision: 1.0000 ± 0.0000 (range: 0.0000)
    Recall:    0.4899 ± 0.0000 (range: 0.0000)
    F1:        0.6576 ± 0.0000 (range: 0.0000)

Model: COMBINED
Loaded model: combined
Best Val Loss: 0.0000
Final Val AUROC: 1.0000

  Testing on C test set...
    Samples: 18673, Positive: 18673
    Accuracy:  0.5326 ± 0.0000 (range: 0.0000)
    Precision: 1.0000 ± 0.0000 (range: 0.0000)
    Recall:    0.5326 ± 0.0000 (range: 0.0000)
    F1:        0.6951 ± 0.